# (Optional) Colab Setup
If you aren't using Colab, you can delete the following code cell. This is just to help students with mounting to Google Drive to access the other .py files and downloading the data, which is a little trickier on Colab than on your local machine using Jupyter. 

In [1]:
# you will be prompted with a window asking to grant permissions
# from google.colab import drive
# drive.mount("/content/drive")

In [2]:
# fill in the path in your Google Drive in the string below. Note: do not escape slashes or spaces
# import os
# datadir = "/content/assignment1"
# if not os.path.exists(datadir):
#   !ln -s "/content/drive/My Drive/YOUR PATH HERE/assignment1/" $datadir
# os.chdir(datadir)
# !pwd

In [3]:
# downloading Fashion-MNIST
# import os
# os.chdir(os.path.join(datadir,"fashion-mnist/"))
# !chmod +x ./get_data.sh
# !./get_data.sh
# os.chdir(datadir)

# Imports

In [4]:
import random
import numpy as np
from data_process import get_FASHION_data, get_RICE_data
# from scipy.spatial import distance
from models import Perceptron, SVM, Softmax, Logistic
from kaggle_submission import output_submission_csv
%matplotlib inline

# For auto-reloading external modules
# See http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

# Loading Fashion-MNIST

In the following cells we determine the number of images for each split and load the images.
<br /> 
TRAIN_IMAGES + VAL_IMAGES = (0, 60000]
, TEST_IMAGES = 10000

In [5]:
# You can change these numbers for experimentation
# For submission we will use the default values 
TRAIN_IMAGES = 50000
VAL_IMAGES = 10000
normalize = True

In [6]:
data = get_FASHION_data(TRAIN_IMAGES, VAL_IMAGES, normalize=normalize)
X_train_fashion, y_train_fashion = data['X_train'], data['y_train']
X_val_fashion, y_val_fashion = data['X_val'], data['y_val']
X_test_fashion, y_test_fashion = data['X_test'], data['y_test']
n_class_fashion = len(np.unique(y_test_fashion))

# Loading Rice

In [7]:
# loads train / test / val splits of 80%, 20%, 20% 
data = get_RICE_data()
X_train_RICE, y_train_RICE = data['X_train'], data['y_train']
X_val_RICE, y_val_RICE = data['X_val'], data['y_val']
X_test_RICE, y_test_RICE = data['X_test'], data['y_test']
n_class_RICE = len(np.unique(y_test_RICE))

print("Number of train samples: ", X_train_RICE.shape[0])
print("Number of val samples: ", X_val_RICE.shape[0])
print("Number of test samples: ", X_test_RICE.shape[0])

Number of train samples:  10911
Number of val samples:  3637
Number of test samples:  3637


### Get Accuracy

This function computes how well your model performs using accuracy as a metric.

In [8]:
def get_acc(pred, y_test):
    return np.sum(y_test == pred) / len(y_test) * 100

# Perceptron

Perceptron has 2 hyperparameters, "learning rate" and "number of epochs" that you can experiment with:
### Hyperparameter 1: Learning rate
The learning rate controls how much we change the current weights of the classifier during each update. We set it at a default value of 0.5, but you should experiment with different values. Here is a guide to help you find a right learning rate: 
- Try values ranging from 0.0005 to 5.0 to see the impact on model accuracy. 
- If the loss oscillates a lot or the loss diverges, the learning rate may be too high. Try decreasing it by a factor of 10 (e.g. from 0.5 to 0.05). 
- If the loss decreases very slowly, the learning rate may be too low and the training can take a long time. Try increasing it by a factor of 10.
- You can also try adding a learning rate decay to slowly reduce the learning rate over each training epoch. For example, multiply the learning rate by 0.95 after each epoch.
- Plot training and validation accuracy over epochs for different learning rates. This will help you visualize the impact of the learning rate.
- [Here](https://towardsdatascience.com/https-medium-com-dashingaditya-rakhecha-understanding-learning-rate-dd5da26bb6de) is a detailed guide to learning rate.

### Hyperparameter 2: Number of Epochs
- An epoch is a complete iterative pass over the training dataset. During an epoch we predict a label using the classifier and then update the weights of the classifier according to the perceptron update rule for each sample in the training set. 
- By observing the training and validation accuracy graphs, you can determine if the model has converged or not. Try increasing the number of epochs if the model is not converging or has low accuracy. 
- You should try different values for the number of training epochs and report your results.

In this section, you will implement the Perceptron classifier in the **models/perceptron.py** script.

The following code: 
- Creates an instance of the Perceptron classifier class 
- An instance of the Perceptron class is trained on the training data using its train method.
- We use the predict method to find the training accuracy as well as the testing accuracy


## Train Perceptron on Fashion-MNIST

In [9]:
results = {}
for lr in [0.0005, 0.005, 0.05, 0.1, 0.2, 0.5, 0.7, 1.0, 2.0, 5.0]:
    for n_epochs in range(30, 101, 10):
        for decay in (True, False):
            percept_fashion = Perceptron(n_class_fashion, lr, n_epochs, decay)
            percept_fashion.train(X_train_fashion, y_train_fashion)
            pred = percept_fashion.predict(X_test_fashion)
            results[lr, n_epochs, decay] = {"accuracy": get_acc(pred, y_test_fashion)}
            print(f"Accuracy for lr={lr}, n_epochs={n_epochs}, decay={decay}: {get_acc(pred, y_test_fashion)}")

top_3 = sorted(results.items(), key=lambda x: x[1]["accuracy"], reverse=True)[:3]
print("Top 3 hyperparameter combinations:")
for params, result in top_3:
    print(f"lr={params[0]}, n_epochs={params[1]}, decay={params[2]}: accuracy={result['accuracy']}")

print(top_3)
best_params, best_result = top_3[0]

percept_fashion = Perceptron(n_class_fashion, best_params[0], best_params[1], best_params[2])
percept_fashion.train(X_train_fashion, y_train_fashion)

Accuracy for lr=0.0005, n_epochs=30, decay=True: 79.02
Accuracy for lr=0.0005, n_epochs=30, decay=False: 78.73
Accuracy for lr=0.0005, n_epochs=40, decay=True: 78.75
Accuracy for lr=0.0005, n_epochs=40, decay=False: 78.75999999999999
Accuracy for lr=0.0005, n_epochs=50, decay=True: 78.64999999999999
Accuracy for lr=0.0005, n_epochs=50, decay=False: 78.57
Accuracy for lr=0.0005, n_epochs=60, decay=True: 78.63
Accuracy for lr=0.0005, n_epochs=60, decay=False: 79.27
Accuracy for lr=0.0005, n_epochs=70, decay=True: 78.94
Accuracy for lr=0.0005, n_epochs=70, decay=False: 78.36999999999999
Accuracy for lr=0.0005, n_epochs=80, decay=True: 78.81
Accuracy for lr=0.0005, n_epochs=80, decay=False: 76.91
Accuracy for lr=0.0005, n_epochs=90, decay=True: 79.09
Accuracy for lr=0.0005, n_epochs=90, decay=False: 78.13
Accuracy for lr=0.0005, n_epochs=100, decay=True: 79.28
Accuracy for lr=0.0005, n_epochs=100, decay=False: 79.11
Accuracy for lr=0.005, n_epochs=30, decay=True: 78.24
Accuracy for lr=0.00

In [10]:
# Best achieved accuracy: 
# l_rate = 0.5
# n_epochs = 50
# decay = False

# percept_fashion = Perceptron(n_class_fashion, l_rate, n_epochs, decay)
# percept_fashion.train(X_train_fashion, y_train_fashion)


pred_percept = percept_fashion.predict(X_train_fashion)
print('The training accuracy is given by: %f' % (get_acc(pred_percept, y_train_fashion)))

The training accuracy is given by: 81.672000


### Validate Perceptron on Fashion-MNIST

In [11]:
pred_percept = percept_fashion.predict(X_val_fashion)
print('The validation accuracy is given by: %f' % (get_acc(pred_percept, y_val_fashion)))

The validation accuracy is given by: 79.040000


### Test Perceptron on Fashion-MNIST

In [12]:
pred_percept = percept_fashion.predict(X_test_fashion)
print('The testing accuracy is given by: %f' % (get_acc(pred_percept, y_test_fashion)))

The testing accuracy is given by: 77.630000


### Perceptron_Fashion-MNIST Kaggle Submission

Once you are satisfied with your solution and test accuracy, output a file to submit your test set predictions to the Kaggle for Assignment 1 Fashion-MNIST. Use the following code to do so:

In [13]:
output_submission_csv('kaggle/perceptron_submission_fashion.csv', percept_fashion.predict(X_test_fashion))

## Train Perceptron on Rice

In [14]:
results = {}
for lr in [0.0005, 0.005, 0.05, 0.1, 0.2, 0.5, 0.7, 1.0, 2.0, 5.0]:
    for n_epochs in range(30, 101, 10):
        for decay in (True, False):
            percept_RICE = Perceptron(n_class_RICE, lr, n_epochs, decay)
            percept_RICE.train(X_train_RICE, y_train_RICE)
            pred = percept_RICE.predict(X_test_RICE)
            results[lr, n_epochs, decay] = {"accuracy": get_acc(pred, y_test_RICE)}
            print(f"Accuracy for lr={lr}, n_epochs={n_epochs}, decay={decay}: {get_acc(pred, y_test_RICE)}")


top_3 = sorted(results.items(), key=lambda x: x[1]["accuracy"], reverse=True)[:3]
print("Top 3 hyperparameter combinations:")
for params, result in top_3:
    print(f"lr={params[0]}, n_epochs={params[1]}, decay={params[2]}: accuracy={result['accuracy']}")

print(top_3)
best_params, best_result = top_3[0]

percept_fashion = Perceptron(n_class_fashion, best_params[0], best_params[1], best_params[2])
percept_fashion.train(X_train_fashion, y_train_fashion)


Accuracy for lr=0.0005, n_epochs=30, decay=True: 99.67005773989553
Accuracy for lr=0.0005, n_epochs=30, decay=False: 98.40527907616166
Accuracy for lr=0.0005, n_epochs=40, decay=True: 99.78003849326367
Accuracy for lr=0.0005, n_epochs=40, decay=False: 99.69755292823756
Accuracy for lr=0.0005, n_epochs=50, decay=True: 99.78003849326367
Accuracy for lr=0.0005, n_epochs=50, decay=False: 98.68023095958208
Accuracy for lr=0.0005, n_epochs=60, decay=True: 99.78003849326367
Accuracy for lr=0.0005, n_epochs=60, decay=False: 99.50508660984327
Accuracy for lr=0.0005, n_epochs=70, decay=True: 99.78003849326367
Accuracy for lr=0.0005, n_epochs=70, decay=False: 99.75254330492163
Accuracy for lr=0.0005, n_epochs=80, decay=True: 99.91751443497388
Accuracy for lr=0.0005, n_epochs=80, decay=False: 99.69755292823756
Accuracy for lr=0.0005, n_epochs=90, decay=True: 99.91751443497388
Accuracy for lr=0.0005, n_epochs=90, decay=False: 99.75254330492163
Accuracy for lr=0.0005, n_epochs=100, decay=True: 99.91

In [15]:
# Best achieved accuracy:
# l_rate = 2
# n_epochs = 50
# decay = True

# percept_RICE = Perceptron(n_class_RICE, l_rate, n_epochs, decay)
# percept_RICE.train(X_train_RICE, y_train_RICE)

pred_percept = percept_RICE.predict(X_train_RICE)
print('The training accuracy is given by: %f' % (get_acc(pred_percept, y_train_RICE)))

The training accuracy is given by: 99.844194


### Validate Perceptron on Rice

In [16]:
pred_percept = percept_RICE.predict(X_val_RICE)
print('The validation accuracy is given by: %f' % (get_acc(pred_percept, y_val_RICE)))

The validation accuracy is given by: 99.752543


### Test Perceptron on Rice

In [17]:
pred_percept = percept_RICE.predict(X_test_RICE)
print('The testing accuracy is given by: %f' % (get_acc(pred_percept, y_test_RICE)))

The testing accuracy is given by: 99.725048


# Support Vector Machines (with SGD)

Next, you will implement a "soft margin" SVM. In this formulation you will maximize the margin between positive and negative training examples and penalize margin violations using a hinge loss.

We will optimize the SVM loss using SGD. This means you must compute the loss function with respect to model weights. You will use this gradient to update the model weights.

SVM optimized with SGD has 3 hyperparameters that you can experiment with:
- **Learning rate** - similar to as defined above in Perceptron, this parameter scales by how much the weights are changed according to the calculated gradient update. 
- **Epochs** - similar to as defined above in Perceptron.
- **Regularization constant** - Hyperparameter to determine the strength of regularization. In this case it is a coefficient on the term which maximizes the margin. You could try different values. The default value is set to 0.05.

You will implement the SVM using SGD in the **models/svm.py**

The following code: 
- Creates an instance of the SVM classifier class 
- The train function of the SVM class is trained on the training data
- We use the predict function to find the training accuracy as well as the testing accuracy

## Train SVM on Fashion-MNIST

In [18]:
results = {}
for lr in [0.0005, 0.005, 0.05, 0.5, 5.0]:
    for n_epochs in range(30, 101, 10):
        for reg_const in [0.001, 0.005, 0.05, 0.1, 0.5, 1.0]:
            svm_fashion = SVM(n_class_fashion, lr, n_epochs, reg_const)
            svm_fashion.train(X_train_fashion, y_train_fashion)
            pred = svm_fashion.predict(X_test_fashion)
            results[lr, n_epochs, reg_const] = {"accuracy": get_acc(pred, y_test_fashion)}
            print(f"Accuracy for lr={lr}, n_epochs={n_epochs}, reg_const={reg_const}: {get_acc(pred, y_test_fashion)}")

top_3 = sorted(results.items(), key=lambda x: x[1]["accuracy"], reverse=True)[:3]
print("Top 3 hyperparameter combinations:")
for params, result in top_3:
    print(f"lr={params[0]}, n_epochs={params[1]}, reg_const={params[2]}: accuracy={result['accuracy']}")

print(top_3)
best_params, best_result = top_3[0]

svm_fashion = SVM(n_class_fashion, best_params[0], best_params[1], best_params[2])
svm_fashion.train(X_train_fashion, y_train_fashion)

Accuracy for lr=0.0005, n_epochs=30, reg_const=0.001: 63.44
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.005: 62.050000000000004
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.05: 62.51
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.1: 62.93
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.5: 61.39
Accuracy for lr=0.0005, n_epochs=30, reg_const=1.0: 60.870000000000005
Accuracy for lr=0.0005, n_epochs=40, reg_const=0.001: 62.73
Accuracy for lr=0.0005, n_epochs=40, reg_const=0.005: 63.13999999999999
Accuracy for lr=0.0005, n_epochs=40, reg_const=0.05: 64.01
Accuracy for lr=0.0005, n_epochs=40, reg_const=0.1: 64.16
Accuracy for lr=0.0005, n_epochs=40, reg_const=0.5: 64.83
Accuracy for lr=0.0005, n_epochs=40, reg_const=1.0: 64.71000000000001
Accuracy for lr=0.0005, n_epochs=50, reg_const=0.001: 64.59
Accuracy for lr=0.0005, n_epochs=50, reg_const=0.005: 64.19
Accuracy for lr=0.0005, n_epochs=50, reg_const=0.05: 64.11
Accuracy for lr=0.0005, n_epochs=50, reg_const=0.1: 65.6000000000

In [19]:
# Best achieved accuracy:
# l_rate = 0.5
# n_epochs = 70
# reg_const = 0.005

# svm_fashion = SVM(n_class_fashion, l_rate, n_epochs, reg_const)
# svm_fashion.train(X_train_fashion, y_train_fashion)


pred_svm = svm_fashion.predict(X_train_fashion)
print('The training accuracy is given by: %f' % (get_acc(pred_svm, y_train_fashion)))

The training accuracy is given by: 79.268000


### Validate SVM on Fashion-MNIST

In [20]:
pred_svm = svm_fashion.predict(X_val_fashion)
print('The validation accuracy is given by: %f' % (get_acc(pred_svm, y_val_fashion)))

The validation accuracy is given by: 79.080000


### Test SVM on Fashion-MNIST

In [21]:
pred_svm = svm_fashion.predict(X_test_fashion)
print('The testing accuracy is given by: %f' % (get_acc(pred_svm, y_test_fashion)))

The testing accuracy is given by: 78.510000


### SVM_Fashion-MNIST Kaggle Submission

Once you are satisfied with your solution and test accuracy output a file to submit your test set predictions to the Kaggle for Assignment 1 Fashion-MNIST. Use the following code to do so:

In [22]:
output_submission_csv('kaggle/svm_submission_fashion.csv', svm_fashion.predict(X_test_fashion))

## Train SVM on Rice

In [24]:
# Adjust the following three hyperparameters to achieve the best accuracy: 
results = {}
for lr in [0.0005, 0.005, 0.05, 0.1, 0.3, 0.5, 2.5, 5.0]:
    for n_epochs in range(30, 101, 10):
        for reg_const in [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0]:
            svm_RICE = SVM(n_class_RICE, lr, n_epochs, reg_const)
            svm_RICE.train(X_train_RICE, y_train_RICE)
            pred = svm_RICE.predict(X_test_RICE)
            acc = get_acc(pred, y_test_RICE)
            results[lr, n_epochs, reg_const] = {"accuracy": acc}
            print(f"Accuracy for lr={lr}, n_epochs={n_epochs}, reg_const={reg_const}: {acc}")

top_3 = sorted(results.items(), key=lambda x: x[1]["accuracy"], reverse=True)[:3]
print("Top 3 hyperparameter combinations:")
for params, result in top_3:
    print(f"lr={params[0]}, n_epochs={params[1]}, reg_const={params[2]}: accuracy={result['accuracy']}")
print(top_3)
best_params, best_result = top_3[0]

svm_RICE = SVM(n_class_RICE, best_params[0], best_params[1], best_params[2])
svm_RICE.train(X_train_RICE, y_train_RICE)

Accuracy for lr=0.0005, n_epochs=30, reg_const=0.001: 75.36431124553204
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.005: 68.40802859499587
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.01: 69.17789386857301
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.05: 68.60049491339015
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.1: 65.43854825405553
Accuracy for lr=0.0005, n_epochs=30, reg_const=0.5: 59.14215012372834
Accuracy for lr=0.0005, n_epochs=30, reg_const=1.0: 67.39070662634039
Accuracy for lr=0.0005, n_epochs=40, reg_const=0.001: 69.34286499862525
Accuracy for lr=0.0005, n_epochs=40, reg_const=0.005: 74.64943634863899
Accuracy for lr=0.0005, n_epochs=40, reg_const=0.01: 62.82650536156172
Accuracy for lr=0.0005, n_epochs=40, reg_const=0.05: 68.6279901017322
Accuracy for lr=0.0005, n_epochs=40, reg_const=0.1: 63.183942810008254
Accuracy for lr=0.0005, n_epochs=40, reg_const=0.5: 68.87544679681056
Accuracy for lr=0.0005, n_epochs=40, reg_const=1.0: 74.64943634863899
Accuracy

In [32]:
# These are the best values from the previous cell
lr = 2.5
n_epochs = 80
reg_const = 0.01


svm_RICE = SVM(n_class_RICE, lr, n_epochs, reg_const)
svm_RICE.train(X_train_RICE, y_train_RICE)

pred_svm = svm_RICE.predict(X_train_RICE)
print('The training accuracy is given by: %f' % (get_acc(pred_svm, y_train_RICE)))

The training accuracy is given by: 79.259463


### Validate SVM on Rice

In [34]:
pred_svm = svm_RICE.predict(X_val_RICE)
print('The validation accuracy is given by: %f' % (get_acc(pred_svm, y_val_RICE)))

The validation accuracy is given by: 78.773715


## Test SVM on Rice

In [27]:
pred_svm = svm_RICE.predict(X_test_RICE)
print('The testing accuracy is given by: %f' % (get_acc(pred_svm, y_test_RICE)))

The testing accuracy is given by: 77.701402


# Softmax Classifier (with SGD)

Next, you will train a Softmax classifier. This classifier consists of a linear function of the input data followed by a softmax function which outputs a vector of dimension C (number of classes) for each data point. Each entry of the softmax output vector corresponds to a confidence in one of the C classes, and like a probability distribution, the entries of the output vector sum to 1. We use a cross-entropy loss on this sotmax output to train the model. 

Check the following link as an additional resource on softmax classification: http://cs231n.github.io/linear-classify/#softmax

Once again we will train the classifier with SGD. This means you need to compute the gradients of the softmax cross-entropy loss function according to the weights and update the weights using this gradient. Check the following link to help with implementing the gradient updates: https://deepnotes.io/softmax-crossentropy

The softmax classifier has 3 hyperparameters that you can experiment with:
- **Learning rate** - As above, this controls how much the model weights are updated with respect to their gradient.
- **Number of Epochs** - As described for perceptron.
- **Regularization constant** - Hyperparameter to determine the strength of regularization. In this case, we minimize the L2 norm of the model weights as regularization, so the regularization constant is a coefficient on the L2 norm in the combined cross-entropy and regularization objective.

You will implement a softmax classifier using SGD in the **models/softmax.py**

The following code: 
- Creates an instance of the Softmax classifier class 
- The train function of the Softmax class is trained on the training data
- We use the predict function to find the training accuracy as well as the testing accuracy

## Train Softmax on Fashion-MNIST

In [78]:
# lr = 1
# n_epochs = 50
# reg_const = 0.05

lr_range = [0.0005, 0.005, 0.05, 0.5]
n_epochs_range = range(50, 101, 10)
reg_const_range = [0.005, 0.05, 0.5]
decay_range = (True, False)

accumulator = 0
total = len(lr_range) * len(n_epochs_range) * len(reg_const_range) * len(decay_range)

results = {}
for lr in lr_range:
    for n_epochs in n_epochs_range:
        for reg_const in reg_const_range:
            for decay in decay_range:
                softmax_fashion = Softmax(n_class_fashion, lr, n_epochs, reg_const, decay)
                softmax_fashion.train(X_train_fashion, y_train_fashion)
                pred = softmax_fashion.predict(X_test_fashion)
                acc = get_acc(pred, y_test_fashion)
                results[lr, n_epochs, reg_const, decay] = acc
                accumulator += 1
                print(f"\r{accumulator}/{total}", end="")
                # print(f"Accuracy for lr={lr}, n_epochs={n_epochs}, reg_const={reg_const}, decay={decay}: {acc}")

top_5 = sorted(results.items(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 hyperparameter combinations:")
for params, result in top_5:
    print(f"lr={params[0]}, n_epochs={params[1]}, reg_const={params[2]}, decay={params[3]}: accuracy={result}")

print(top_5)
best_params, best_result = top_5[0]
softmax_fashion = Softmax(n_class_fashion, best_params[0], best_params[1], best_params[3])

# Refer to hints in the calculate_gradient and train methods of the Softmax class in "models/softmax.py"
softmax_fashion.train(X_train_fashion, y_train_fashion)

144/144Top 5 hyperparameter combinations:
lr=0.05, n_epochs=90, reg_const=0.005, decay=False: accuracy=82.12
lr=0.05, n_epochs=80, reg_const=0.05, decay=True: accuracy=81.83
lr=0.0005, n_epochs=70, reg_const=0.5, decay=True: accuracy=81.57
lr=0.005, n_epochs=100, reg_const=0.5, decay=True: accuracy=81.49
lr=0.005, n_epochs=100, reg_const=0.005, decay=False: accuracy=81.47999999999999
[((0.05, 90, 0.005, False), np.float64(82.12)), ((0.05, 80, 0.05, True), np.float64(81.83)), ((0.0005, 70, 0.5, True), np.float64(81.57)), ((0.005, 100, 0.5, True), np.float64(81.49)), ((0.005, 100, 0.005, False), np.float64(81.47999999999999))]


In [84]:
# Best parameters:
# l_rate = 0.05
# n_epochs = 90
# reg_const = 0.005
# decay = False

# softmax_fashion = Softmax(n_class_fashion, l_rate, n_epochs, reg_const, decay)
# softmax_fashion.train(X_train_fashion, y_train_fashion)

pred_softmax = softmax_fashion.predict(X_train_fashion)
print('The training accuracy is given by: %f' % (get_acc(pred_softmax, y_train_fashion)))

The training accuracy is given by: 86.168000


### Validate Softmax on Fashion-MNIST

In [85]:
pred_softmax = softmax_fashion.predict(X_val_fashion)
print('The validation accuracy is given by: %f' % (get_acc(pred_softmax, y_val_fashion)))

The validation accuracy is given by: 80.210000


### Testing Softmax on Fashion-MNIST

In [86]:
pred_softmax = softmax_fashion.predict(X_test_fashion)
print('The testing accuracy is given by: %f' % (get_acc(pred_softmax, y_test_fashion)))

The testing accuracy is given by: 79.930000


### Softmax_Fashion-MNIST Kaggle Submission

Once you are satisfied with your solution and test accuracy output a file to submit your test set predictions to the Kaggle for Assignment 1 Fashion-MNIST. Use the following code to do so:

In [87]:
output_submission_csv('kaggle/softmax_submission_fashion.csv', softmax_fashion.predict(X_test_fashion))

## Train Softmax on Rice

In [92]:
# lr = 0.5
# n_epochs = 10
# reg_const = 0.05

lr_range = [0.0005, 0.005, 0.05, 0.5]
n_epochs_range = range(100, 201, 10)
reg_const_range = [0.005, 0.05, 0.5, 1, 2.5, 5]
decay_range = (True, False)

accumulator = 0
total = len(lr_range) * len(n_epochs_range) * len(reg_const_range) * len(decay_range)

results = {}
for lr in lr_range:
    for n_epochs in n_epochs_range:
        for reg_const in reg_const_range:
            for decay in decay_range:
                softmax_RICE = Softmax(n_class_RICE, lr, n_epochs, reg_const, decay)
                softmax_RICE.train(X_train_RICE, y_train_RICE)
                pred = softmax_RICE.predict(X_test_RICE)
                acc = get_acc(pred, y_test_RICE)
                results[lr, n_epochs, reg_const, decay] = acc
                accumulator += 1
                print(f"\r{accumulator}/{total}", end="")
                # print(f"Accuracy for lr={lr}, n_epochs={n_epochs}, reg_const={reg_const}, decay={decay}: {acc}")

top_5 = sorted(results.items(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 hyperparameter combinations:")
for params, result in top_5:
    print(f"lr={params[0]}, n_epochs={params[1]}, reg_const={params[2]}, decay={params[3]}: accuracy={result}")

print(top_5)
best_params, best_result = top_5[0]

softmax_RICE = Softmax(n_class_RICE, best_params[0], best_params[1], best_params[3])
softmax_RICE.train(X_train_RICE, y_train_RICE)

407/528

/Users/edu/Documents/UIUC/CS444/assignment1/models/softmax.py:41: RuntimeWarning: overflow encountered in subtract
  feature_scores = np.exp(fs - np.max(fs, axis=1, keepdims=True))
/Users/edu/Documents/UIUC/CS444/assignment1/models/softmax.py:41: RuntimeWarning: invalid value encountered in subtract
  feature_scores = np.exp(fs - np.max(fs, axis=1, keepdims=True))


528/528Top 5 hyperparameter combinations:
lr=0.0005, n_epochs=150, reg_const=0.5, decay=True: accuracy=99.89001924663185
lr=0.0005, n_epochs=190, reg_const=0.005, decay=True: accuracy=99.89001924663185
lr=0.0005, n_epochs=100, reg_const=0.05, decay=True: accuracy=99.86252405828981
lr=0.0005, n_epochs=110, reg_const=0.05, decay=True: accuracy=99.86252405828981
lr=0.0005, n_epochs=110, reg_const=0.5, decay=True: accuracy=99.86252405828981
[((0.0005, 150, 0.5, True), np.float64(99.89001924663185)), ((0.0005, 190, 0.005, True), np.float64(99.89001924663185)), ((0.0005, 100, 0.05, True), np.float64(99.86252405828981)), ((0.0005, 110, 0.05, True), np.float64(99.86252405828981)), ((0.0005, 110, 0.5, True), np.float64(99.86252405828981))]


In [93]:
# Best parameters:
# l_rate = 0.0005
# n_epochs = 150
# reg_const = 0.05
# decay = True


pred_softmax = softmax_RICE.predict(X_train_RICE)
print('The training accuracy is given by: %f' % (get_acc(pred_softmax, y_train_RICE)))

The training accuracy is given by: 81.596554


### Validate Softmax on Rice

In [94]:
pred_softmax = softmax_RICE.predict(X_val_RICE)
print('The validation accuracy is given by: %f' % (get_acc(pred_softmax, y_val_RICE)))

The validation accuracy is given by: 81.605719


### Testing Softmax on Rice

In [95]:
pred_softmax = softmax_RICE.predict(X_test_RICE)
print('The testing accuracy is given by: %f' % (get_acc(pred_softmax, y_test_RICE)))

The testing accuracy is given by: 81.825681


# Logistic Classifier

The Logistic Classifier has 2 hyperparameters that you can experiment with:
- **Learning rate** - similar to as defined above in Perceptron, this parameter scales by how much the weights are changed according to the calculated gradient update. 
- **Number of Epochs** - As described for perceptron.
- **Threshold** - The decision boundary of the classifier.


You will implement the Logistic Classifier in the **models/logistic.py**

The following code: 
- Creates an instance of the Logistic classifier class 
- The train function of the Logistic class is trained on the training data
- We use the predict function to find the training accuracy as well as the testing accuracy

### Training Logistic Classifer

In [ ]:
learning_rate = 0.5
n_epochs = 10
threshold = 0.5

lr = Logistic(learning_rate, n_epochs, threshold)

# Refer to hints in the sigmoid and train methods of the Logistic class in "models/logistic.py"
lr.train(X_train_RICE, y_train_RICE)

In [ ]:
pred_lr = lr.predict(X_train_RICE)
print('The training accuracy is given by: %f' % (get_acc(pred_lr, y_train_RICE)))

### Validate Logistic Classifer

In [ ]:
pred_lr = lr.predict(X_val_RICE)
print('The validation accuracy is given by: %f' % (get_acc(pred_lr, y_val_RICE)))

### Test Logistic Classifier

In [ ]:
pred_lr = lr.predict(X_test_RICE)
print('The testing accuracy is given by: %f' % (get_acc(pred_lr, y_test_RICE)))